# Gap 2: CIPT vs FOPT — Does $\mathcal{N}_\lambda$ Interact Differently?

**Background.** $R_\tau$ can be computed two ways:
- **FOPT** (Fixed-Order PT): expand $\alpha_s$ at $m_\tau^2$, integrate the contour analytically.
- **CIPT** (Contour-Improved PT): integrate $\alpha_s(s)$ numerically around $|s| = m_\tau^2$.

CIPT resums running-coupling effects along the complex contour. There is a
long-standing discrepancy between the two: they give different $\alpha_s(m_\tau)$
extractions ($\sim 2\sigma$ apart). This is an actual open problem.

**Question:** Does $\mathcal{N}_\lambda$ attenuation of $\beta_0$ interact differently
with CIPT's contour resummation vs FOPT's fixed expansion?

**Key formula:**
$$\delta^{(0)} = \frac{1}{2\pi i} \oint_{|s|=m_\tau^2} \frac{ds}{s} \left(1 - 2\frac{s}{m_\tau^2} + 2\frac{s^3}{m_\tau^6} - \frac{s^4}{m_\tau^8}\right) \Pi(s)$$

where $\Pi(s) = \sum K_n (\alpha_s(-s)/\pi)^n$ with $\alpha_s$ running along the contour.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.graded_scn import (
    beta0_standard, beta0_attenuated, alpha_s_running_1loop,
    r_tau_coefficients, ALPHA_S_MZ, M_Z, M_TAU,
    R_TAU_EXPERIMENT, R_TAU_UNCERTAINTY, V_UD_SQ, S_EW, N_C
)

plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True})

In [ ]:
# ── 1-loop running in the complex plane ───────────────────────
def alpha_s_complex_1loop(s_complex, nf, beta_0):
    """1-loop αs(s) for complex s.
    
    αs(s) = αs(M_Z²) / (1 + β_0 αs(M_Z²)/(4π) ln(s/M_Z²))
    
    For s = m_τ² e^{iθ}, ln(s/M_Z²) = ln(m_τ²/M_Z²) + iθ
    """
    log_ratio = np.log(s_complex / M_Z**2)  # complex log
    denom = 1.0 + (beta_0 * ALPHA_S_MZ) / (4 * np.pi) * log_ratio
    return ALPHA_S_MZ / denom

# ── FOPT: fixed-order at m_τ² ─────────────────────────────────
def delta_fopt(alpha_s_mtau, max_order=4):
    """δ^(0) in FOPT = Σ K_n (αs(m_τ)/π)^n."""
    K = r_tau_coefficients()
    a = alpha_s_mtau / np.pi
    return sum(K[n] * a**n for n in range(1, max_order + 1))

# ── CIPT: contour integration ─────────────────────────────────
def delta_cipt(nf, beta_0, max_order=4, n_theta=1000):
    """δ^(0) in CIPT via numerical contour integration.
    
    δ^(0) = (1/2π) ∫_0^{2π} dθ (1 + 2e^{iθ} + ...) × Σ K_n (αs(m_τ² e^{iθ})/π)^n
    
    The weight function for the V+A correlator:
    w(θ) = (1 + 2e^{iθ} - 2e^{3iθ} - e^{4iθ}) from the kinematic factor.
    
    Using the standard CIPT contour integral moments:
    J_n = (1/2π) ∫_0^{2π} dθ w(e^{iθ}) (αs(m_τ² e^{iθ})/π)^n
    δ^(0)_CIPT = Σ K_n J_n
    """
    K = r_tau_coefficients()
    
    theta = np.linspace(0, 2 * np.pi, n_theta, endpoint=False)
    dtheta = 2 * np.pi / n_theta
    
    delta = 0.0
    terms = {}
    
    for n in range(1, max_order + 1):
        # Compute J_n = (1/2π) ∫ w(e^{iθ}) (αs(m_τ² e^{iθ})/π)^n dθ
        integrand = np.zeros(n_theta, dtype=complex)
        
        for i, th in enumerate(theta):
            s = M_TAU**2 * np.exp(1j * th)
            a_s = alpha_s_complex_1loop(s, nf, beta_0)
            
            # Weight function w(x) = 1 + 2x - 2x³ - x⁴ where x = e^{iθ}
            x = np.exp(1j * th)
            w = 1 + 2*x - 2*x**3 - x**4
            
            integrand[i] = w * (a_s / np.pi)**n
        
        J_n = np.sum(integrand).real * dtheta / (2 * np.pi)
        terms[n] = K[n] * J_n
        delta += K[n] * J_n
    
    return delta, terms

# ── Compare FOPT vs CIPT at standard β_0 ─────────────────────
nf = 3
b0_std = beta0_standard(nf)
as_mtau = alpha_s_running_1loop(M_TAU**2, nf, b0_std)

print(f"β_0(nf=3) = {b0_std}")
print(f"αs(m_τ) = {as_mtau:.4f}")
print()

d_fopt = delta_fopt(as_mtau)
d_cipt, cipt_terms = delta_cipt(nf, b0_std)

r_tau_tree = N_C * V_UD_SQ * S_EW

print(f"δ_pert (FOPT):  {d_fopt:.6f}")
print(f"δ_pert (CIPT):  {d_cipt:.6f}")
print(f"R_τ (FOPT):     {r_tau_tree * (1 + d_fopt):.4f}")
print(f"R_τ (CIPT):     {r_tau_tree * (1 + d_cipt):.4f}")
print(f"R_τ (exp):      {R_TAU_EXPERIMENT}")
print()
print("CIPT term-by-term:")
K = r_tau_coefficients()
for n in sorted(cipt_terms):
    fopt_term = K[n] * (as_mtau/np.pi)**n
    print(f"  n={n}: CIPT = {cipt_terms[n]:.6f}  FOPT = {fopt_term:.6f}  ratio = {cipt_terms[n]/fopt_term:.4f}")

In [ ]:
# ── Attenuation scan: FOPT vs CIPT response to λ ─────────────
print("Does N_λ affect CIPT differently than FOPT?")
print()
print(f"{'λ':>5}  {'β_0':>6}  {'αs(mτ)':>8}  {'δ_FOPT':>10}  {'δ_CIPT':>10}  {'R_FOPT':>8}  {'R_CIPT':>8}  {'CIPT-FOPT':>10}")
print("-" * 80)

lam_vals = [0.8, 0.9, 1.0, 1.1, 1.2, 1.5, 2.0, 3.0, 5.0]
fopt_vals = []
cipt_vals = []

for lam in lam_vals:
    b0 = beta0_attenuated(nf, lam)
    a_s = alpha_s_running_1loop(M_TAU**2, nf, b0)
    
    if np.isnan(a_s):
        print(f"{lam:>5.1f}  {b0:>6.2f}  {'nan':>8}")
        continue
    
    d_f = delta_fopt(a_s)
    d_c, _ = delta_cipt(nf, b0)
    
    r_f = r_tau_tree * (1 + d_f)
    r_c = r_tau_tree * (1 + d_c)
    
    fopt_vals.append((lam, r_f))
    cipt_vals.append((lam, r_c))
    
    print(f"{lam:>5.1f}  {b0:>6.2f}  {a_s:>8.4f}  {d_f:>10.6f}  {d_c:>10.6f}  {r_f:>8.4f}  {r_c:>8.4f}  {d_c-d_f:>10.6f}")

print()
print(f"R_τ (exp) = {R_TAU_EXPERIMENT} ± {R_TAU_UNCERTAINTY}")

In [ ]:
# ── Find optimal λ for each method ────────────────────────────
lam_scan = np.linspace(0.5, 5.0, 200)
r_fopt_scan = []
r_cipt_scan = []

for lam in lam_scan:
    b0 = beta0_attenuated(nf, lam)
    a_s = alpha_s_running_1loop(M_TAU**2, nf, b0)
    if np.isnan(a_s):
        r_fopt_scan.append(np.nan)
        r_cipt_scan.append(np.nan)
        continue
    r_fopt_scan.append(r_tau_tree * (1 + delta_fopt(a_s)))
    d_c, _ = delta_cipt(nf, b0, n_theta=500)
    r_cipt_scan.append(r_tau_tree * (1 + d_c))

r_fopt_scan = np.array(r_fopt_scan)
r_cipt_scan = np.array(r_cipt_scan)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: R_τ vs λ
ax1.plot(lam_scan, r_fopt_scan, 'b-', label='FOPT', linewidth=2)
ax1.plot(lam_scan, r_cipt_scan, 'r-', label='CIPT', linewidth=2)
ax1.axhline(R_TAU_EXPERIMENT, color='green', linestyle='--', alpha=0.7, label=f'Exp = {R_TAU_EXPERIMENT}')
ax1.fill_between(lam_scan, R_TAU_EXPERIMENT - R_TAU_UNCERTAINTY, R_TAU_EXPERIMENT + R_TAU_UNCERTAINTY,
                  alpha=0.1, color='green')
ax1.axvline(1.0, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('λ (attenuation parameter)')
ax1.set_ylabel(r'$R_\tau$')
ax1.set_title(r'$R_\tau$ vs $\beta_0$ attenuation')
ax1.legend()

# Right: CIPT - FOPT difference vs λ
diff = r_cipt_scan - r_fopt_scan
ax2.plot(lam_scan, diff, 'k-', linewidth=2)
ax2.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax2.axvline(1.0, color='gray', linestyle=':', alpha=0.5)
ax2.set_xlabel('λ')
ax2.set_ylabel(r'$R_\tau^{CIPT} - R_\tau^{FOPT}$')
ax2.set_title('FOPT/CIPT discrepancy vs attenuation')

plt.tight_layout()
plt.show()

# Does attenuation resolve the FOPT/CIPT discrepancy?
mask = ~np.isnan(r_fopt_scan) & ~np.isnan(r_cipt_scan)
diffs = np.abs(r_cipt_scan[mask] - r_fopt_scan[mask])
diff_at_1 = abs(r_cipt_scan[np.argmin(np.abs(lam_scan - 1.0))] - r_fopt_scan[np.argmin(np.abs(lam_scan - 1.0))])
print(f"FOPT/CIPT difference at λ=1: {diff_at_1:.4f}")
print(f"Minimum difference:          {np.min(diffs):.4f} at λ={lam_scan[mask][np.argmin(diffs)]:.2f}")
print(f"Maximum difference:          {np.max(diffs):.4f}")

## Conclusions

Does $\beta_0$ attenuation interact differently with CIPT vs FOPT?

The key diagnostic is the CIPT−FOPT difference as a function of $\lambda$.
If attenuation narrows the gap, it could resolve the FOPT/CIPT discrepancy.
If it doesn't, or makes it worse, this is another null result.